# Apple Game Rank Reranker Colab

Drive의 `rank_data.bin`으로 ranking reranker를 학습하고, 로컬 Go/ONNX Runtime에서 바로 읽을 수 있는 **단일 `.onnx` 파일**을 만듭니다.

준비물:

- 로컬에서 생성한 `rank_data.bin`
- Drive 경로: `/content/drive/MyDrive/apple_game/rank_data.bin`
- Colab Runtime: GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, glob, subprocess, textwrap

GITHUB_URL = "https://github.com/scheveningen361/10_apple_game.git"
REPO_DIR = "/content/10_apple_game"
DRIVE_DIR = "/content/drive/MyDrive/apple_game"
DATA_NAME = "rank_data.bin"

MODEL_STEM = "model_rank_small_colab"
PT_PATH = f"models/{MODEL_STEM}.pt"
ONNX_PATH = f"models/{MODEL_STEM}.onnx"

os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive dir:", DRIVE_DIR)

In [ ]:
if os.path.exists(os.path.join(REPO_DIR, ".git")):
    %cd {REPO_DIR}
    !git fetch origin
    !git reset --hard origin/master
else:
    %cd /content
    !git clone {GITHUB_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -5

In [ ]:
!pip -q install -U onnx onnxscript
import torch, onnx, onnxscript
print("torch:", torch.__version__)
print("onnx:", onnx.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
src = os.path.join(DRIVE_DIR, DATA_NAME)
dst = os.path.join(REPO_DIR, "data/generated", DATA_NAME)
os.makedirs(os.path.dirname(dst), exist_ok=True)

if not os.path.exists(src):
    raise FileNotFoundError(f"Drive에 데이터가 없습니다: {src}")

if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
    shutil.copy2(src, dst)

print("data:", dst, f"{os.path.getsize(dst)/1e6:.1f} MB")

## Small 모델 학습

로컬 CPU에서 검증한 작은 모델입니다. GPU에서는 15 epoch가 빠르게 끝납니다.

이미 Drive에 `.pt`가 있고 이어서 학습하려면 아래 셀 대신 resume 셀을 사용하세요.

In [ ]:
!python rl/train_rank.py \
  --data data/generated/rank_data.bin \
  --out {PT_PATH} \
  --export {ONNX_PATH} \
  --epochs 15 \
  --groups-per-batch 128 \
  --channels 32 \
  --blocks 2

## Resume 학습용

Drive에 저장된 기존 `.pt`를 가져와서 30 epoch까지 이어서 학습할 때 사용합니다. 필요할 때만 실행하세요.

In [ ]:
drive_pt = os.path.join(DRIVE_DIR, os.path.basename(PT_PATH))
if not os.path.exists(drive_pt):
    raise FileNotFoundError(f"resume checkpoint가 Drive에 없습니다: {drive_pt}")
os.makedirs("models", exist_ok=True)
shutil.copy2(drive_pt, PT_PATH)
print("resume from", PT_PATH)

In [ ]:
!python rl/train_rank.py \
  --data data/generated/rank_data.bin \
  --out {PT_PATH} \
  --resume {PT_PATH} \
  --epochs 30 \
  --groups-per-batch 128 \
  --channels 32 \
  --blocks 2 \
  --export {ONNX_PATH}

## Export-only

학습은 끝났는데 export만 다시 하고 싶을 때 사용합니다. `external_data=False`로 export되므로 `.onnx.data` 파일이 생기지 않아야 합니다.

In [ ]:
!python rl/train_rank.py \
  --out {PT_PATH} \
  --export {ONNX_PATH} \
  --export-only

In [ ]:
onnx_related = sorted(glob.glob(ONNX_PATH + "*"))
print("ONNX files:", onnx_related)
if any(p.endswith(".data") for p in onnx_related):
    raise RuntimeError("external .onnx.data 파일이 생겼습니다. train_rank.py의 external_data=False 적용 여부를 확인하세요.")

for path in [PT_PATH, ONNX_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    dst = os.path.join(DRIVE_DIR, os.path.basename(path))
    shutil.copy2(path, dst)
    print("copied:", dst, f"{os.path.getsize(dst)/1e6:.1f} MB")

## 로컬 평가 명령

Drive에서 `model_rank_small_colab.onnx` 하나만 로컬 `models/` 폴더로 내려받으면 됩니다.

```powershell
go run -tags nn ./cmd/apple-game `
  -nn-rerank `
  -n 100 `
  -model models/model_rank_small_colab.onnx `
  -ort-lib runtime/onnxruntime.dll `
  -rerank-weight 0.25 `
  -rerank-k 8
```

큰 손실이 아직 보이면 `-rerank-weight 0.15`도 비교하세요.